# Builder Agent — user perspective

**What it does.** Implements the scoped change for one Linear work item. Read/Edit/Write source files; run `pytest`/`ruff`/`mypy`/`python` for validation. **No** git, **no** Linear, **no** PR creation (BLDR-04).

**Entry point.** `run_builder_agent(input: BuilderInput) -> BuilderOutput`.

**Schema guard.** `BuilderOutput` rejects extra fields like `git_branch`/`pr_url` — even if the LLM emits them, the validator strips the output and forces a retry.

**Cost.** Live runs touch the local filesystem (your scratch repo) and consume tokens.

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Construct the input

The caller MUST fetch fresh Linear state immediately before constructing `BuilderInput` (Pitfall 6). Pass the issue body, ACs, and the absolute repo root the Builder should treat as its `cwd`.

In [ ]:
from hsb.contracts.builder import BuilderInput, RepositoryContext

example_input = BuilderInput(
    work_item_id="LIN-456",
    issue_description="Add a docstring to scratch.py:greet().",
    acceptance_criteria=[
        "scratch.py:greet has a one-line docstring",
        "ruff check passes on scratch.py",
    ],
    plan_source="docs/plan.md",
    repository_context=RepositoryContext(
        root_path="/tmp/hsb-builder-scratch",
        technical_stack=["python"],
    ),
)
print(example_input.model_dump_json(indent=2))

## Live invocation

Gates: `HSB_NOTEBOOK_RUN_LIVE=1` AND `HSB_NOTEBOOK_SCRATCH_DIR` pointing at a real local repo. The Builder will Read/Edit files there — pick a throwaway directory.

In [ ]:
import os

from _helpers import assert_g1_safe, gated, live_mode

scratch = os.environ.get("HSB_NOTEBOOK_SCRATCH_DIR")
if not live_mode() or not scratch:
    print(gated("Builder live run (set HSB_NOTEBOOK_SCRATCH_DIR=/tmp/throwaway-repo)"))
else:
    assert_g1_safe()
    from hsb.agents.builder_agent import run_builder_agent

    inp = BuilderInput(
        work_item_id="LIN-NB-1",
        issue_description="Add a one-line docstring to scratch.py:greet().",
        acceptance_criteria=["greet() has a docstring"],
        plan_source="plan.md",
        repository_context=RepositoryContext(
            root_path=scratch, technical_stack=["python"]
        ),
    )
    out = run_builder_agent(inp)
    print("status:", out.implementation_status)
    print("files changed:", [f.path for f in out.files_changed])
    print("validation:", out.validation.model_dump())